<a href="https://colab.research.google.com/github/leon-asim/EEG-Attention/blob/main/inssiok_eeg.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import kagglehub
import os

# Download latest version
path = kagglehub.dataset_download("inancigdem/eeg-data-for-mental-attention-state-detection")

path = os.path.join(path, 'EEG Data')

print("Path to dataset files:", path)

100%|██████████| 557M/557M [00:05<00:00, 115MB/s]

Extracting files...


Path to dataset files: /root/.cache/kagglehub/datasets/inancigdem/eeg-data-for-mental-attention-state-detection/versions/1/EEG Data


In [2]:
!pip install scipy

In [3]:
from scipy.io import loadmat
import numpy as np
import pandas as pd
import scipy.io
import matplotlib.pyplot as plt
from tqdm import tqdm

from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.metrics import f1_score, precision_score, recall_score

import torch
from torch.utils.data import Dataset, DataLoader, random_split
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import torchvision.transforms as transforms

In [4]:
data_root = path
files = os.listdir(data_root)
len(files)

34

In [5]:
FOCUSED_CLASS = 0
UNFOCUSED_CLASS = 1
DROWNSY_CLASS = 2

In [6]:
columns = [
    'ED_COUNTER',    'ED_INTERPOLATED',    'ED_RAW_CQ',    'ED_AF3',    'ED_F7',
    'ED_F3',    'ED_FC5',    'ED_T7',    'ED_P7',    'ED_O1',
    'ED_O2',    'ED_P8',    'ED_T8',    'ED_FC6',    'ED_F4',
    'ED_F8',    'ED_AF4',    'ED_GYROX',    'ED_GYROY',    'ED_TIMESTAMP',
    'ED_ES_TIMESTAMP',    'ED_FUNC_ID',    'ED_FUNC_VALUE',    'ED_MARKER',    'ED_SYNC_SIGNAL'
]

def get_state(timestamp):
    if timestamp <= 10*128*60:
        return FOCUSED_CLASS
    elif timestamp > 20*128*60:
        return UNFOCUSED_CLASS
    else:
        return DROWNSY_CLASS

# Scale data
scaler = StandardScaler()

#  Function to read in the EEG data and extract the valid lead data.
#  Returns a dataframe.
def get_EEG_data(data_root, filename):
    hz = 128
    mat = scipy.io.loadmat(os.path.join(data_root, filename))
    data = mat["o"]["data"][0,0]
    eeg_df = pd.DataFrame(data, columns=columns)
    eeg_df = eeg_df.filter(['ED_AF3', 'ED_F7', 'ED_F3', 'ED_FC5',
                            'ED_T7', 'ED_P7', 'ED_O1', 'ED_O2',
                            'ED_P8', 'ED_T8', 'ED_FC6', 'ED_F4',
                            'ED_F8', 'ED_AF4'])
    labels = ['AF3','F7', 'F3','FC5','T7','P7','O1','O2','P8','T8', 'FC6','F4','F8','AF4']
    eeg_df.columns = labels
    eeg_df = pd.DataFrame(scaler.fit_transform(eeg_df), columns=eeg_df.columns)
    eeg_df.reset_index(inplace=True)
    eeg_df.rename(columns={'index': 'timestamp'}, inplace=True)

    eeg_df['state'] = eeg_df['timestamp'].apply(get_state)

    return eeg_df

In [7]:
dataset = []
# For each file, print # minutes of data
for filename in files:
    data = get_EEG_data(data_root, filename)
    dataset.append(data)

In [8]:
dataset[0]

,timestamp,AF3,F7,F3,FC5,T7,P7,O1,O2,P8,T8,FC6,F4,F8,AF4,state
0,0,-0.331954,0.077346,-0.709276,-0.122137,-1.054267,0.175925,2.204488,-0.862589,-1.729453,-0.970781,-1.646813,1.408173,-0.211862,-0.085991,0
1,1,-0.256578,0.207288,-0.658741,1.189750,-4.514545,0.217030,2.300048,-0.683522,-1.566128,-0.970781,-1.085650,1.082847,-0.122167,-0.043210,0
2,2,0.170551,0.223531,-0.603152,0.483349,-4.322308,0.266356,2.545776,-0.618407,-1.384655,-0.970781,-0.384197,2.384151,-0.839728,-0.007559,0
3,3,0.245926,0.223531,-0.587991,-1.232195,0.483634,0.274577,2.627685,-0.667243,-1.366508,-0.970781,0.457547,2.709477,-0.122167,-0.014689,0
4,4,-0.080702,0.385958,-0.552617,-0.323966,-1.438743,0.291019,2.654988,-0.471898,-1.420950,-0.970781,0.457547,1.896162,0.416003,0.006702,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
360923,360923,0.798681,-0.945948,0.144771,3.409865,1.060347,0.044389,-1.945581,-2.799764,-1.566128,1.484497,-1.225941,-1.194434,0.505699,-0.132337,1
360924,360924,0.949433,-1.075891,0.205413,3.208037,0.099158,0.040278,-1.918278,-2.848601,-1.675011,0.368462,0.738129,-0.218456,0.236613,-0.160858,1
360925,360925,0.371552,-0.945948,0.099289,0.786092,-0.093079,0.032057,-1.918278,-2.604419,-1.602422,1.484497,2.141036,0.106870,0.146918,-0.160858,1
360926,360926,0.095175,-0.978434,0.008325,1.391578,-0.285317,0.027947,-1.863671,-2.392795,-1.529833,4.832605,3.684234,0.269533,0.146918,-0.171553,1


In [9]:
print((dataset[0].state == 2).sum())

76800


In [10]:
type(dataset), type(dataset[0])

(list, pandas.core.frame.DataFrame)

In [11]:
print(len(files))
print(files)

34
['eeg_record24.mat', 'eeg_record17.mat', 'eeg_record33.mat', 'eeg_record11.mat', 'eeg_record10.mat', 'eeg_record22.mat', 'eeg_record19.mat', 'eeg_record31.mat', 'eeg_record26.mat', 'eeg_record14.mat', 'eeg_record23.mat', 'eeg_record29.mat', 'eeg_record32.mat', 'eeg_record3.mat', 'eeg_record1.mat', 'eeg_record13.mat', 'eeg_record16.mat', 'eeg_record2.mat', 'eeg_record25.mat', 'eeg_record30.mat', 'eeg_record12.mat', 'eeg_record18.mat', 'eeg_record8.mat', 'eeg_record20.mat', 'eeg_record6.mat', 'eeg_record15.mat', 'eeg_record34.mat', 'eeg_record9.mat', 'eeg_record7.mat', 'eeg_record5.mat', 'eeg_record21.mat', 'eeg_record4.mat', 'eeg_record28.mat', 'eeg_record27.mat']


In [12]:
from sklearn.model_selection import train_test_split

34 different experiments and we are going to split them 70/15/15 for training, validating and testing.

In [13]:
# Fixed order
files = sorted(files)

# 70% train, 30% temporary
train_files, temp_files = train_test_split(
    files,
    test_size=0.30,
    random_state=42
)

# Split remaining 30% into 15% validation + 15% test
val_files, test_files = train_test_split(
    temp_files,
    test_size=0.50,
    random_state=42
)

print("Train:", len(train_files))
print("Validation:", len(val_files))
print("Test:", len(test_files))

print("\nTrain subjects:", train_files)
print("\nValidation subjects:", val_files)
print("\nTest subjects:", test_files)

Train: 23
Validation: 5
Test: 6

Train subjects: ['eeg_record13.mat', 'eeg_record24.mat', 'eeg_record25.mat', 'eeg_record14.mat', 'eeg_record21.mat', 'eeg_record2.mat', 'eeg_record10.mat', 'eeg_record11.mat', 'eeg_record6.mat', 'eeg_record12.mat', 'eeg_record5.mat', 'eeg_record30.mat', 'eeg_record7.mat', 'eeg_record3.mat', 'eeg_record26.mat', 'eeg_record32.mat', 'eeg_record15.mat', 'eeg_record28.mat', 'eeg_record9.mat', 'eeg_record16.mat', 'eeg_record19.mat', 'eeg_record22.mat', 'eeg_record4.mat']

Validation subjects: ['eeg_record8.mat', 'eeg_record17.mat', 'eeg_record20.mat', 'eeg_record33.mat', 'eeg_record29.mat']

Test subjects: ['eeg_record31.mat', 'eeg_record23.mat', 'eeg_record18.mat', 'eeg_record1.mat', 'eeg_record34.mat', 'eeg_record27.mat']


In [14]:
def split_epochs(data, hz=128, epoch_length=2, step_size=1):
    epoch_samples = int(epoch_length * hz)  # 256
    step_samples = int(step_size * hz)     # 128

    epochs = []

    for start in range(0, len(data) - epoch_samples + 1, step_samples):
        epoch = data.iloc[start:start + epoch_samples].copy()

        # Don't allow an epoch to contain multiple states
        if epoch['state'].nunique() == 1:
            epochs.append(epoch)

    return epochs

In [15]:
MAX_EPOCHS_PER_SUBJECT = 1000

def create_epochs(file_list):
    all_epochs = []

    for filename in file_list:
        print("Processing:", filename)

        data = get_EEG_data(data_root, filename)
        epochs = split_epochs(data)

        # Randomly select at most 1000 epochs
        if len(epochs) > MAX_EPOCHS_PER_SUBJECT:
            rng = np.random.default_rng(42)
            indices = rng.choice(
                len(epochs),
                MAX_EPOCHS_PER_SUBJECT,
                replace=False
            )
            epochs = [epochs[i] for i in indices]

        all_epochs.extend(epochs)

    return all_epochs

In [16]:
train_epochs = create_epochs(train_files)
val_epochs = create_epochs(val_files)
test_epochs = create_epochs(test_files)

print("Train epochs:", len(train_epochs))
print("Validation epochs:", len(val_epochs))
print("Test epochs:", len(test_epochs))

Processing: eeg_record13.mat
Processing: eeg_record24.mat
Processing: eeg_record25.mat
Processing: eeg_record14.mat
Processing: eeg_record21.mat
Processing: eeg_record2.mat
Processing: eeg_record10.mat
Processing: eeg_record11.mat
Processing: eeg_record6.mat
Processing: eeg_record12.mat
Processing: eeg_record5.mat
Processing: eeg_record30.mat
Processing: eeg_record7.mat
Processing: eeg_record3.mat
Processing: eeg_record26.mat
Processing: eeg_record32.mat
Processing: eeg_record15.mat
Processing: eeg_record28.mat
Processing: eeg_record9.mat
Processing: eeg_record16.mat
Processing: eeg_record19.mat
Processing: eeg_record22.mat
Processing: eeg_record4.mat
Processing: eeg_record8.mat
Processing: eeg_record17.mat
Processing: eeg_record20.mat
Processing: eeg_record33.mat
Processing: eeg_record29.mat
Processing: eeg_record31.mat
Processing: eeg_record23.mat
Processing: eeg_record18.mat
Processing: eeg_record1.mat
Processing: eeg_record34.mat
Processing: eeg_record27.mat
Train epochs: 23000
Val

In [17]:
class EEGDataset(Dataset):
    def __init__(self, dataframes, target_column='state'):
        self.data = []
        self.targets = []

        for df in dataframes:
            self.targets.append(int(df[target_column].iloc[0]))

            feature = df.drop(
                columns=[target_column, 'timestamp'],
                errors='ignore'
            )

            self.data.append(feature.values)

        self.data = torch.tensor(
            np.array(self.data),
            dtype=torch.float32
        )

        self.targets = torch.tensor(
            self.targets,
            dtype=torch.long
        )

    def __len__(self):
        return len(self.targets)

    def __getitem__(self, idx):
        return self.data[idx], self.targets[idx]

In [18]:
train_dataset = EEGDataset(train_epochs)
val_dataset = EEGDataset(val_epochs)
test_dataset = EEGDataset(test_epochs)

In [19]:
train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=32,
    shuffle=False
)

test_loader = DataLoader(
    test_dataset,
    batch_size=32,
    shuffle=False
)

In [20]:
print(train_dataset.data.shape)
print(train_dataset.targets.shape)

print(val_dataset.data.shape)
print(test_dataset.data.shape)

torch.Size([23000, 256, 14])
torch.Size([23000])
torch.Size([5000, 256, 14])
torch.Size([6000, 256, 14])


In [21]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

class BiLSTM(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers, num_classes):
        super().__init__()

        self.hidden_size = hidden_size
        self.num_layers = num_layers

        self.lstm = nn.LSTM(
            input_size,
            hidden_size,
            num_layers,
            batch_first=True,
            bidirectional=True
        )

        self.fc1 = nn.Linear(hidden_size * 2, hidden_size)
        self.fc2 = nn.Linear(hidden_size, hidden_size // 2)
        self.fc3 = nn.Linear(hidden_size // 2, num_classes)

        self.dropout = nn.Dropout(0.5)

    def forward(self, x):
        batch_size = x.size(0)

        h0 = torch.zeros(
            self.num_layers * 2,
            batch_size,
            self.hidden_size,
            device=x.device
        )

        c0 = torch.zeros(
            self.num_layers * 2,
            batch_size,
            self.hidden_size,
            device=x.device
        )

        out, _ = self.lstm(x, (h0, c0))

        out = F.relu(self.fc1(out[:, -1, :]))
        out = F.relu(self.fc2(out))
        out = self.fc3(self.dropout(out))

        return out

## 14 Channels

In [22]:
model = BiLSTM(
    input_size=14,
    hidden_size=256,
    num_layers=2,
    num_classes=3
).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

In [23]:
from collections import Counter

print("Train:", Counter(train_dataset.targets.numpy()))
print("Validation:", Counter(val_dataset.targets.numpy()))
print("Test:", Counter(test_dataset.targets.numpy()))

Train: Counter({np.int64(1): 12954, np.int64(0): 5103, np.int64(2): 4943})
Validation: Counter({np.int64(1): 3209, np.int64(0): 938, np.int64(2): 853})
Test: Counter({np.int64(1): 3573, np.int64(0): 1228, np.int64(2): 1199})


In [24]:
from sklearn.utils.class_weight import compute_class_weight
import numpy as np
import torch

In [25]:
classes = np.array([0, 1, 2])

weights = compute_class_weight(
    class_weight='balanced',
    classes=classes,
    y=train_dataset.targets.numpy()
)

class_weights = torch.tensor(
    weights,
    dtype=torch.float32
).to(device)

print(class_weights)

tensor([1.5024, 0.5918, 1.5510], device='cuda:0')


In [26]:
criterion = nn.CrossEntropyLoss(weight=class_weights)

In [27]:
from sklearn.metrics import f1_score
from tqdm import tqdm
import copy

10 epochs for now, for more accurate results the number of epochs will be increased

In [28]:
num_epochs = 10
best_val_loss = float("inf")
best_model_state = None

train_losses = []
val_losses = []
train_f1s = []
val_f1s = []

for epoch in range(num_epochs):

    # -------- TRAIN --------
    model.train()
    running_loss = 0
    train_preds = []
    train_labels = []

    for data, targets in tqdm(train_loader, leave=False):
        data = data.to(device)
        targets = targets.to(device)

        optimizer.zero_grad()

        outputs = model(data)
        loss = criterion(outputs, targets)

        loss.backward()
        optimizer.step()

        running_loss += loss.item()

        preds = outputs.argmax(dim=1)
        train_preds.extend(preds.cpu().numpy())
        train_labels.extend(targets.cpu().numpy())

    train_loss = running_loss / len(train_loader)
    train_f1 = f1_score(
        train_labels,
        train_preds,
        average="macro"
    )

    # -------- VALIDATION --------
    model.eval()
    running_loss = 0
    val_preds = []
    val_labels = []

    with torch.no_grad():
        for data, targets in val_loader:
            data = data.to(device)
            targets = targets.to(device)

            outputs = model(data)
            loss = criterion(outputs, targets)

            running_loss += loss.item()

            preds = outputs.argmax(dim=1)
            val_preds.extend(preds.cpu().numpy())
            val_labels.extend(targets.cpu().numpy())

    val_loss = running_loss / len(val_loader)
    val_f1 = f1_score(
        val_labels,
        val_preds,
        average="macro"
    )

    train_losses.append(train_loss)
    val_losses.append(val_loss)
    train_f1s.append(train_f1)
    val_f1s.append(val_f1)

    print(
        f"Epoch {epoch+1}/{num_epochs} | "
        f"Train Loss: {train_loss:.4f} | "
        f"Train F1: {train_f1:.4f} | "
        f"Val Loss: {val_loss:.4f} | "
        f"Val F1: {val_f1:.4f}"
    )

    # Save best model
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_model_state = copy.deepcopy(model.state_dict())

Epoch 1/10 | Train Loss: 0.8503 | Train F1: 0.6110 | Val Loss: 0.9920 | Val F1: 0.5444


Epoch 2/10 | Train Loss: 0.8464 | Train F1: 0.5925 | Val Loss: 0.9157 | Val F1: 0.4413


Epoch 3/10 | Train Loss: 0.8171 | Train F1: 0.6022 | Val Loss: 0.8258 | Val F1: 0.5803


Epoch 4/10 | Train Loss: 0.7477 | Train F1: 0.6558 | Val Loss: 0.7673 | Val F1: 0.6147


Epoch 5/10 | Train Loss: 0.6786 | Train F1: 0.6946 | Val Loss: 0.7540 | Val F1: 0.6078


Epoch 6/10 | Train Loss: 0.6348 | Train F1: 0.7120 | Val Loss: 0.7384 | Val F1: 0.6425


Epoch 7/10 | Train Loss: 0.6354 | Train F1: 0.7059 | Val Loss: 0.8792 | Val F1: 0.5668


Epoch 8/10 | Train Loss: 0.6432 | Train F1: 0.7042 | Val Loss: 0.7800 | Val F1: 0.6092


Epoch 9/10 | Train Loss: 0.6167 | Train F1: 0.7164 | Val Loss: 0.9121 | Val F1: 0.5256


Epoch 10/10 | Train Loss: 0.5751 | Train F1: 0.7315 | Val Loss: 0.7962 | Val F1: 0.5886


In [31]:
model.load_state_dict(best_model_state)

<All keys matched successfully>

In [29]:
from sklearn.metrics import classification_report, confusion_matrix

In [33]:
model.eval()

test_preds = []
test_labels = []

with torch.no_grad():
    for data, targets in test_loader:
        data = data.to(device)

        outputs = model(data)
        preds = outputs.argmax(dim=1)

        test_preds.extend(preds.cpu().numpy())
        test_labels.extend(targets.numpy())

print(classification_report(
    test_labels,
    test_preds,
    target_names=["Focused", "Unfocused", "Drowsy"]
))

print(confusion_matrix(test_labels, test_preds))

              precision    recall  f1-score   support

     Focused       0.81      0.57      0.67      1228
   Unfocused       0.78      0.87      0.82      3573
      Drowsy       0.25      0.24      0.24      1199

    accuracy                           0.68      6000
   macro avg       0.61      0.56      0.58      6000
weighted avg       0.68      0.68      0.67      6000

[[ 699   85  444]
 [  63 3111  399]
 [  97  817  285]]


## 8 Channels

In [ ]:
channels_8 = ['AF3', 'F7', 'F3', 'FC5',
              'FC6', 'F4', 'F8', 'AF4']

In [ ]:
def reduce_channels(dataset, channels):
    indices = [dataset.data.shape[2] for _ in []]  # ignore

In [ ]:
def select_channels(epochs, channels):
    return [
        epoch[['timestamp'] + channels + ['state']]
        for epoch in epochs
    ]

In [ ]:
def select_channels(epochs, channels):
    return [
        epoch[['timestamp'] + channels + ['state']]
        for epoch in epochs
    ]

train_8 = select_channels(train_epochs, channels_8)
val_8 = select_channels(val_epochs, channels_8)
test_8 = select_channels(test_epochs, channels_8)

train_dataset_8 = EEGDataset(train_8)
val_dataset_8 = EEGDataset(val_8)
test_dataset_8 = EEGDataset(test_8)

In [ ]:
train_loader = DataLoader(train_dataset_8, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset_8, batch_size=32)
test_loader = DataLoader(test_dataset_8, batch_size=32)

In [ ]:
model = BiLSTM(
    input_size=8,
    hidden_size=256,
    num_layers=2,
    num_classes=3
).to(device)

criterion = nn.CrossEntropyLoss(weight=class_weights)

optimizer = optim.Adam(
    model.parameters(),
    lr=0.001
)

In [ ]:
num_epochs = 10
best_val_loss = float("inf")
best_model_state = None

train_losses = []
val_losses = []
train_f1s = []
val_f1s = []

for epoch in range(num_epochs):

    # -------- TRAIN --------
    model.train()
    running_loss = 0
    train_preds = []
    train_labels = []

    for data, targets in tqdm(train_loader, leave=False):
        data = data.to(device)
        targets = targets.to(device)

        optimizer.zero_grad()

        outputs = model(data)
        loss = criterion(outputs, targets)

        loss.backward()
        optimizer.step()

        running_loss += loss.item()

        preds = outputs.argmax(dim=1)
        train_preds.extend(preds.cpu().numpy())
        train_labels.extend(targets.cpu().numpy())

    train_loss = running_loss / len(train_loader)
    train_f1 = f1_score(
        train_labels,
        train_preds,
        average="macro"
    )

    # -------- VALIDATION --------
    model.eval()
    running_loss = 0
    val_preds = []
    val_labels = []

    with torch.no_grad():
        for data, targets in val_loader:
            data = data.to(device)
            targets = targets.to(device)

            outputs = model(data)
            loss = criterion(outputs, targets)

            running_loss += loss.item()

            preds = outputs.argmax(dim=1)
            val_preds.extend(preds.cpu().numpy())
            val_labels.extend(targets.cpu().numpy())

    val_loss = running_loss / len(val_loader)
    val_f1 = f1_score(
        val_labels,
        val_preds,
        average="macro"
    )

    train_losses.append(train_loss)
    val_losses.append(val_loss)
    train_f1s.append(train_f1)
    val_f1s.append(val_f1)

    print(
        f"Epoch {epoch+1}/{num_epochs} | "
        f"Train Loss: {train_loss:.4f} | "
        f"Train F1: {train_f1:.4f} | "
        f"Val Loss: {val_loss:.4f} | "
        f"Val F1: {val_f1:.4f}"
    )

    # Save best model
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_model_state = copy.deepcopy(model.state_dict())

Epoch 1/10 | Train Loss: 0.9035 | Train F1: 0.5641 | Val Loss: 0.8884 | Val F1: 0.5564


Epoch 2/10 | Train Loss: 0.8094 | Train F1: 0.6228 | Val Loss: 0.8714 | Val F1: 0.4871


Epoch 3/10 | Train Loss: 0.7728 | Train F1: 0.6391 | Val Loss: 0.8875 | Val F1: 0.5286


Epoch 4/10 | Train Loss: 0.7548 | Train F1: 0.6457 | Val Loss: 0.8201 | Val F1: 0.5919


Epoch 5/10 | Train Loss: 0.7317 | Train F1: 0.6581 | Val Loss: 0.8818 | Val F1: 0.5179


Epoch 6/10 | Train Loss: 0.6939 | Train F1: 0.6708 | Val Loss: 0.8941 | Val F1: 0.5522


Epoch 7/10 | Train Loss: 0.6545 | Train F1: 0.6898 | Val Loss: 0.9521 | Val F1: 0.5363


Epoch 8/10 | Train Loss: 0.6214 | Train F1: 0.7032 | Val Loss: 0.9498 | Val F1: 0.5473


Epoch 9/10 | Train Loss: 0.5767 | Train F1: 0.7279 | Val Loss: 0.8846 | Val F1: 0.5600


Epoch 10/10 | Train Loss: 0.5201 | Train F1: 0.7532 | Val Loss: 1.1628 | Val F1: 0.4991


In [ ]:
model.eval()

test_preds = []
test_labels = []

with torch.no_grad():
    for data, targets in test_loader:
        data = data.to(device)

        outputs = model(data)
        preds = outputs.argmax(dim=1)

        test_preds.extend(preds.cpu().numpy())
        test_labels.extend(targets.numpy())

print(classification_report(
    test_labels,
    test_preds,
    target_names=["Focused", "Unfocused", "Drowsy"]
))

print(confusion_matrix(test_labels, test_preds))

              precision    recall  f1-score   support

     Focused       0.61      0.70      0.66      1228
   Unfocused       0.90      0.46      0.60      3573
      Drowsy       0.28      0.65      0.39      1199

    accuracy                           0.54      6000
   macro avg       0.60      0.60      0.55      6000
weighted avg       0.72      0.54      0.57      6000

[[ 865   59  304]
 [ 238 1626 1709]
 [ 304  120  775]]


In [ ]:
print(train_dataset_8.data.shape)
print(train_dataset_8.targets.shape)

print(torch.unique(
    train_dataset_8.targets,
    return_counts=True
))

torch.Size([23000, 256, 8])
torch.Size([23000])
(tensor([0, 1, 2]), tensor([ 5103, 12954,  4943]))


In [ ]:
print(model)

BiLSTM(
  (lstm): LSTM(8, 256, num_layers=2, batch_first=True, bidirectional=True)
  (fc1): Linear(in_features=512, out_features=256, bias=True)
  (fc2): Linear(in_features=256, out_features=128, bias=True)
  (fc3): Linear(in_features=128, out_features=3, bias=True)
  (dropout): Dropout(p=0.5, inplace=False)
)


In [ ]:
print(train_dataset_8.data.min().item())
print(train_dataset_8.data.max().item())
print(train_dataset_8.data.mean().item())
print(train_dataset_8.data.std().item())

-373.71368408203125
243.5138702392578
-0.00018931103113573045
0.9858754873275757


In [ ]:
print(train_dataset.data.min().item())
print(train_dataset.data.max().item())
print(train_dataset.data.mean().item())
print(train_dataset.data.std().item())

-373.71368408203125
243.5138702392578
-0.0008185346960090101
0.9835011959075928


In [ ]:
print(torch.isnan(train_dataset_8.data).any())
print(torch.isinf(train_dataset_8.data).any())

tensor(False)
tensor(False)


In [ ]:
data, targets = next(iter(train_loader))

data = data.to(device)
targets = targets.to(device)

model.train()

outputs = model(data)
loss = criterion(outputs, targets)

optimizer.zero_grad()
loss.backward()

print("Loss:", loss.item())
print("Gradient:", model.lstm.weight_ih_l0.grad.abs().mean().item())

Loss: 0.7097591161727905
Gradient: 0.00023448940191883594


## 4 Channels

In [ ]:
channels_4 = ['AF3', 'F3', 'F4', 'AF4']

In [ ]:
train_4 = select_channels(train_epochs, channels_4)
val_4 = select_channels(val_epochs, channels_4)
test_4 = select_channels(test_epochs, channels_4)

train_dataset_4 = EEGDataset(train_4)
val_dataset_4 = EEGDataset(val_4)
test_dataset_4 = EEGDataset(test_4)

In [ ]:
train_loader = DataLoader(train_dataset_4, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset_4, batch_size=32, shuffle=False)
test_loader = DataLoader(test_dataset_4, batch_size=32, shuffle=False)

In [ ]:
print(train_dataset_4.data.shape)

torch.Size([23000, 256, 4])


In [ ]:
model = BiLSTM(
    input_size=4,
    hidden_size=256,
    num_layers=2,
    num_classes=3
).to(device)

criterion = nn.CrossEntropyLoss(weight=class_weights)

optimizer = optim.Adam(
    model.parameters(),
    lr=0.001
)

In [ ]:
num_epochs = 10
best_val_loss = float("inf")
best_model_state = None

train_losses = []
val_losses = []
train_f1s = []
val_f1s = []

for epoch in range(num_epochs):

    # -------- TRAIN --------
    model.train()
    running_loss = 0
    train_preds = []
    train_labels = []

    for data, targets in tqdm(train_loader, leave=False):
        data = data.to(device)
        targets = targets.to(device)

        optimizer.zero_grad()

        outputs = model(data)
        loss = criterion(outputs, targets)

        loss.backward()
        optimizer.step()

        running_loss += loss.item()

        preds = outputs.argmax(dim=1)
        train_preds.extend(preds.cpu().numpy())
        train_labels.extend(targets.cpu().numpy())

    train_loss = running_loss / len(train_loader)
    train_f1 = f1_score(
        train_labels,
        train_preds,
        average="macro"
    )

    # -------- VALIDATION --------
    model.eval()
    running_loss = 0
    val_preds = []
    val_labels = []

    with torch.no_grad():
        for data, targets in val_loader:
            data = data.to(device)
            targets = targets.to(device)

            outputs = model(data)
            loss = criterion(outputs, targets)

            running_loss += loss.item()

            preds = outputs.argmax(dim=1)
            val_preds.extend(preds.cpu().numpy())
            val_labels.extend(targets.cpu().numpy())

    val_loss = running_loss / len(val_loader)
    val_f1 = f1_score(
        val_labels,
        val_preds,
        average="macro"
    )

    train_losses.append(train_loss)
    val_losses.append(val_loss)
    train_f1s.append(train_f1)
    val_f1s.append(val_f1)

    print(
        f"Epoch {epoch+1}/{num_epochs} | "
        f"Train Loss: {train_loss:.4f} | "
        f"Train F1: {train_f1:.4f} | "
        f"Val Loss: {val_loss:.4f} | "
        f"Val F1: {val_f1:.4f}"
    )

    # Save best model
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_model_state = copy.deepcopy(model.state_dict())

Epoch 1/10 | Train Loss: 0.9818 | Train F1: 0.5077 | Val Loss: 0.9954 | Val F1: 0.4453


Epoch 2/10 | Train Loss: 0.9559 | Train F1: 0.5174 | Val Loss: 1.0051 | Val F1: 0.3988


Epoch 3/10 | Train Loss: 0.9319 | Train F1: 0.5323 | Val Loss: 0.9617 | Val F1: 0.5004


Epoch 4/10 | Train Loss: 0.9209 | Train F1: 0.5410 | Val Loss: 0.9893 | Val F1: 0.4805


Epoch 5/10 | Train Loss: 0.9107 | Train F1: 0.5511 | Val Loss: 0.9750 | Val F1: 0.4840


Epoch 6/10 | Train Loss: 0.8828 | Train F1: 0.5630 | Val Loss: 0.9815 | Val F1: 0.4584


Epoch 7/10 | Train Loss: 0.8638 | Train F1: 0.5721 | Val Loss: 0.9283 | Val F1: 0.5099


Epoch 8/10 | Train Loss: 0.8362 | Train F1: 0.5802 | Val Loss: 0.9306 | Val F1: 0.5142


Epoch 9/10 | Train Loss: 0.8236 | Train F1: 0.5925 | Val Loss: 0.9301 | Val F1: 0.4945


Epoch 10/10 | Train Loss: 0.8319 | Train F1: 0.5888 | Val Loss: 0.9362 | Val F1: 0.4765


In [ ]:
model.eval()

test_preds = []
test_labels = []

with torch.no_grad():
    for data, targets in test_loader:
        data = data.to(device)

        outputs = model(data)
        preds = outputs.argmax(dim=1)

        test_preds.extend(preds.cpu().numpy())
        test_labels.extend(targets.numpy())

print(classification_report(
    test_labels,
    test_preds,
    target_names=["Focused", "Unfocused", "Drowsy"]
))

print(confusion_matrix(test_labels, test_preds))

              precision    recall  f1-score   support

     Focused       0.59      0.51      0.55      1228
   Unfocused       0.80      0.65      0.72      3573
      Drowsy       0.30      0.51      0.38      1199

    accuracy                           0.60      6000
   macro avg       0.56      0.56      0.55      6000
weighted avg       0.66      0.60      0.62      6000

[[ 630  145  453]
 [ 273 2339  961]
 [ 162  430  607]]


## 2 Channels

In [ ]:
channels_2 = ['AF3', 'AF4']

train_2 = select_channels(train_epochs, channels_2)
val_2 = select_channels(val_epochs, channels_2)
test_2 = select_channels(test_epochs, channels_2)

train_dataset_2 = EEGDataset(train_2)
val_dataset_2 = EEGDataset(val_2)
test_dataset_2 = EEGDataset(test_2)

train_loader = DataLoader(train_dataset_2, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset_2, batch_size=32, shuffle=False)
test_loader = DataLoader(test_dataset_2, batch_size=32, shuffle=False)

In [ ]:
model = BiLSTM(
    input_size=2,
    hidden_size=256,
    num_layers=2,
    num_classes=3
).to(device)

criterion = nn.CrossEntropyLoss(weight=class_weights)

optimizer = optim.Adam(model.parameters(), lr=0.001)

In [ ]:
num_epochs = 10
best_val_loss = float("inf")
best_model_state = None

train_losses = []
val_losses = []
train_f1s = []
val_f1s = []

for epoch in range(num_epochs):

    # -------- TRAIN --------
    model.train()
    running_loss = 0
    train_preds = []
    train_labels = []

    for data, targets in tqdm(train_loader, leave=False):
        data = data.to(device)
        targets = targets.to(device)

        optimizer.zero_grad()

        outputs = model(data)
        loss = criterion(outputs, targets)

        loss.backward()
        optimizer.step()

        running_loss += loss.item()

        preds = outputs.argmax(dim=1)
        train_preds.extend(preds.cpu().numpy())
        train_labels.extend(targets.cpu().numpy())

    train_loss = running_loss / len(train_loader)
    train_f1 = f1_score(
        train_labels,
        train_preds,
        average="macro"
    )

    # -------- VALIDATION --------
    model.eval()
    running_loss = 0
    val_preds = []
    val_labels = []

    with torch.no_grad():
        for data, targets in val_loader:
            data = data.to(device)
            targets = targets.to(device)

            outputs = model(data)
            loss = criterion(outputs, targets)

            running_loss += loss.item()

            preds = outputs.argmax(dim=1)
            val_preds.extend(preds.cpu().numpy())
            val_labels.extend(targets.cpu().numpy())

    val_loss = running_loss / len(val_loader)
    val_f1 = f1_score(
        val_labels,
        val_preds,
        average="macro"
    )

    train_losses.append(train_loss)
    val_losses.append(val_loss)
    train_f1s.append(train_f1)
    val_f1s.append(val_f1)

    print(
        f"Epoch {epoch+1}/{num_epochs} | "
        f"Train Loss: {train_loss:.4f} | "
        f"Train F1: {train_f1:.4f} | "
        f"Val Loss: {val_loss:.4f} | "
        f"Val F1: {val_f1:.4f}"
    )

    # Save best model
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_model_state = copy.deepcopy(model.state_dict())

Epoch 1/10 | Train Loss: 1.0384 | Train F1: 0.4130 | Val Loss: 1.0711 | Val F1: 0.3511


Epoch 2/10 | Train Loss: 1.0033 | Train F1: 0.4614 | Val Loss: 0.9477 | Val F1: 0.5016


Epoch 3/10 | Train Loss: 0.9827 | Train F1: 0.4969 | Val Loss: 1.0607 | Val F1: 0.4160


Epoch 4/10 | Train Loss: 1.0182 | Train F1: 0.4636 | Val Loss: 1.1045 | Val F1: 0.1971


Epoch 5/10 | Train Loss: 0.9436 | Train F1: 0.5224 | Val Loss: 1.0238 | Val F1: 0.4061


Epoch 6/10 | Train Loss: 0.9134 | Train F1: 0.5434 | Val Loss: 0.9764 | Val F1: 0.4265


Epoch 7/10 | Train Loss: 0.9058 | Train F1: 0.5366 | Val Loss: 0.9365 | Val F1: 0.4419


Epoch 8/10 | Train Loss: 0.8896 | Train F1: 0.5445 | Val Loss: 0.9213 | Val F1: 0.4913


Epoch 9/10 | Train Loss: 0.8840 | Train F1: 0.5484 | Val Loss: 0.9312 | Val F1: 0.4494


Epoch 10/10 | Train Loss: 0.8693 | Train F1: 0.5637 | Val Loss: 0.9418 | Val F1: 0.4666


In [ ]:
model.eval()

test_preds = []
test_labels = []

with torch.no_grad():
    for data, targets in test_loader:
        data = data.to(device)

        outputs = model(data)
        preds = outputs.argmax(dim=1)

        test_preds.extend(preds.cpu().numpy())
        test_labels.extend(targets.numpy())

print(classification_report(
    test_labels,
    test_preds,
    target_names=["Focused", "Unfocused", "Drowsy"]
))

print(confusion_matrix(test_labels, test_preds))

              precision    recall  f1-score   support

     Focused       0.48      0.68      0.56      1228
   Unfocused       0.88      0.27      0.41      3573
      Drowsy       0.25      0.67      0.37      1199

    accuracy                           0.43      6000
   macro avg       0.54      0.54      0.45      6000
weighted avg       0.67      0.43      0.43      6000

[[ 837   48  343]
 [ 594  965 2014]
 [ 317   81  801]]


## 1 Channel

In [ ]:
channels_1 = ['AF3']

In [ ]:
train_1 = select_channels(train_epochs, channels_1)
val_1 = select_channels(val_epochs, channels_1)
test_1 = select_channels(test_epochs, channels_1)

train_dataset_1 = EEGDataset(train_1)
val_dataset_1 = EEGDataset(val_1)
test_dataset_1 = EEGDataset(test_1)

train_loader = DataLoader(train_dataset_1, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset_1, batch_size=32, shuffle=False)
test_loader = DataLoader(test_dataset_1, batch_size=32, shuffle=False)

In [ ]:
model = BiLSTM(
    input_size=1,
    hidden_size=256,
    num_layers=2,
    num_classes=3
).to(device)

criterion = nn.CrossEntropyLoss(weight=class_weights)

optimizer = optim.Adam(model.parameters(), lr=0.001)

In [ ]:
num_epochs = 10
best_val_loss = float("inf")
best_model_state = None

train_losses = []
val_losses = []
train_f1s = []
val_f1s = []

for epoch in range(num_epochs):

    # -------- TRAIN --------
    model.train()
    running_loss = 0
    train_preds = []
    train_labels = []

    for data, targets in tqdm(train_loader, leave=False):
        data = data.to(device)
        targets = targets.to(device)

        optimizer.zero_grad()

        outputs = model(data)
        loss = criterion(outputs, targets)

        loss.backward()
        optimizer.step()

        running_loss += loss.item()

        preds = outputs.argmax(dim=1)
        train_preds.extend(preds.cpu().numpy())
        train_labels.extend(targets.cpu().numpy())

    train_loss = running_loss / len(train_loader)
    train_f1 = f1_score(
        train_labels,
        train_preds,
        average="macro"
    )

    # -------- VALIDATION --------
    model.eval()
    running_loss = 0
    val_preds = []
    val_labels = []

    with torch.no_grad():
        for data, targets in val_loader:
            data = data.to(device)
            targets = targets.to(device)

            outputs = model(data)
            loss = criterion(outputs, targets)

            running_loss += loss.item()

            preds = outputs.argmax(dim=1)
            val_preds.extend(preds.cpu().numpy())
            val_labels.extend(targets.cpu().numpy())

    val_loss = running_loss / len(val_loader)
    val_f1 = f1_score(
        val_labels,
        val_preds,
        average="macro"
    )

    train_losses.append(train_loss)
    val_losses.append(val_loss)
    train_f1s.append(train_f1)
    val_f1s.append(val_f1)

    print(
        f"Epoch {epoch+1}/{num_epochs} | "
        f"Train Loss: {train_loss:.4f} | "
        f"Train F1: {train_f1:.4f} | "
        f"Val Loss: {val_loss:.4f} | "
        f"Val F1: {val_f1:.4f}"
    )

    # Save best model
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_model_state = copy.deepcopy(model.state_dict())

Epoch 1/10 | Train Loss: 1.0348 | Train F1: 0.4397 | Val Loss: 1.0393 | Val F1: 0.4127


Epoch 2/10 | Train Loss: 0.9915 | Train F1: 0.4780 | Val Loss: 1.0105 | Val F1: 0.4146


Epoch 3/10 | Train Loss: 0.9609 | Train F1: 0.5077 | Val Loss: 0.9582 | Val F1: 0.4709


Epoch 4/10 | Train Loss: 0.9956 | Train F1: 0.4743 | Val Loss: 1.0657 | Val F1: 0.4203


Epoch 5/10 | Train Loss: 0.9972 | Train F1: 0.4828 | Val Loss: 1.0257 | Val F1: 0.4394


Epoch 6/10 | Train Loss: 0.9473 | Train F1: 0.5149 | Val Loss: 0.9167 | Val F1: 0.5204


Epoch 7/10 | Train Loss: 0.9181 | Train F1: 0.5351 | Val Loss: 0.9181 | Val F1: 0.5136


Epoch 8/10 | Train Loss: 0.9231 | Train F1: 0.5305 | Val Loss: 0.9105 | Val F1: 0.5079


Epoch 9/10 | Train Loss: 0.9068 | Train F1: 0.5419 | Val Loss: 0.9187 | Val F1: 0.4614


Epoch 10/10 | Train Loss: 0.8950 | Train F1: 0.5464 | Val Loss: 0.9039 | Val F1: 0.4854


In [ ]:
model.eval()

test_preds = []
test_labels = []

with torch.no_grad():
    for data, targets in test_loader:
        data = data.to(device)

        outputs = model(data)
        preds = outputs.argmax(dim=1)

        test_preds.extend(preds.cpu().numpy())
        test_labels.extend(targets.numpy())

print(classification_report(
    test_labels,
    test_preds,
    target_names=["Focused", "Unfocused", "Drowsy"]
))

print(confusion_matrix(test_labels, test_preds))

              precision    recall  f1-score   support

     Focused       0.51      0.54      0.53      1228
   Unfocused       0.88      0.52      0.66      3573
      Drowsy       0.30      0.66      0.42      1199

    accuracy                           0.55      6000
   macro avg       0.57      0.57      0.53      6000
weighted avg       0.69      0.55      0.58      6000

[[ 664   86  478]
 [ 385 1872 1316]
 [ 247  166  786]]
